# Topic: Number Systems - IEEE 754 Float standard, float epsilon precision loss, and struct binary conversions
 
## 1. NUMBER SYSTEMS & DIGITAL REPRESENTATION
- **Binary (Base-2)**, **Decimal (Base-10)**, and **Hexadecimal (Base-16)** represent numerical values using different radix bases.
- **CPython Memory Layout**:
  - **Integers**: Python 3 integers have arbitrary precision (dynamically sized). They are represented by a C-struct containing an array of digit digits, allowing infinite growth (subject to memory limits).
  - **Floats**: Python floats are standard C double-precision floating-point numbers (64-bit), conforming to the **IEEE 754 standard**.
 
## 2. IEEE 754 DOUBLE-PRECISION STANDARD
- A 64-bit float is partitioned into:
  1. **Sign bit**: 1 bit (0 for positive, 1 for negative).
  2. **Exponent**: 11 bits (represents scale, biased by 1023).
  3. **Fraction (Mantissa/Significand)**: 52 bits (represents precision digits).
- Value mapping formula:
  $$\text{Value} = (-1)^{\text{sign}} \times \left(1 + \sum_{i=1}^{52} b_{52-i} 2^{-i}\right) \times 2^{\text{exponent} - 1023}$$
 
## 3. PRECISION LOSS & FLOATING-POINT EPSILON
- **Floating-Point Precision Loss**:
  - Certain decimal fractions (like `0.1` or `0.2`) cannot be represented exactly as finite binary fractions. This leads to slight representation errors.
  - Classic symptom: `0.1 + 0.2 != 0.3` (returns `0.30000000000000004`).
- **Machine Epsilon ($\epsilon$)**:
  - The upper bound on relative error due to rounding in floating-point arithmetic.
  - In 64-bit Python floats, this is exposed as `sys.float_info.epsilon` (value $\approx 2.22 \times 10^{-16}$).
  - When comparing float equality, you should check if the difference is smaller than a small tolerance (epsilon) rather than using direct `==`:
    `abs(a - b) < epsilon`
 
---


In [ ]:
import sys
import struct

# 1. Floating-Point Precision Loss validation
print("--- Floating-Point Precision Loss ---")
sum_val = 0.1 + 0.2
print(f"0.1 + 0.2 value: {sum_val:.25f}")
print(f"Is 0.1 + 0.2 == 0.3? {sum_val == 0.3}")  # Expected: False

# 2. Correct float comparison using Epsilon
epsilon = sys.float_info.epsilon
print(f"\nMachine Epsilon (sys.float_info.epsilon): {epsilon}")
print(f"Is abs(sum_val - 0.3) < epsilon? {abs(sum_val - 0.3) < epsilon}")  # Expected: True



In [ ]:
# 3. Dynamic binary bit extraction of Float objects (IEEE 754)
def float_to_binary_bits(val):
    """Converts a float to its 64-bit IEEE 754 binary string representation using struct."""
    # Pack float into 8 bytes (double precision 'd') and unpack as 64-bit unsigned int ('Q')
    [packed_int] = struct.unpack('Q', struct.pack('d', val))
    # Format as 64-character binary string
    binary_str = f"{packed_int:064b}"
    
    sign = binary_str[0]
    exponent = binary_str[1:12]
    mantissa = binary_str[12:]
    return sign, exponent, mantissa

print("\n--- IEEE 754 Binary Representation ---")
for num in [1.0, 0.5, 0.1]:
    s, e, m = float_to_binary_bits(num)
    print(f"Value: {num:>3} | Sign: {s} | Exponent (11b): {e} | Mantissa (52b): {m[:10]}...")
